# PULSE - Motor Predictivo | Exploración y Experimentación
**Notebook complementario a los scripts de producción**

Este notebook es para exploración interactiva:
- Análisis del dataset y target
- Experimentación con features
- Visualización de resultados
- Debug de casos específicos

Los scripts `.py` son los de producción. Acá se experimenta antes de incorporar cambios.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import shap
import joblib

from pathlib import Path
import sys
sys.path.append('..')  # Para importar config y scripts

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

print('✓ Imports OK')

## 1. Cargar Dataset Procesado

In [ ]:
from config import OUTPUTS_DIR, MODELS_DIR, TARGET_COL

# Cargar dataset procesado (output de 01_build_dataset.py)
df = pd.read_csv(OUTPUTS_DIR / 'dataset_processed.csv', parse_dates=['fecha_creacion'])
print(f'Dataset: {df.shape}')
df.head(3)

## 2. Análisis del Target

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribución global
target_counts = df[TARGET_COL].value_counts()
axes[0].pie(target_counts, labels=['Cumplió', 'Incumplió'], autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'])
axes[0].set_title('Distribución del Target')

# Evolución temporal
if 'fecha_creacion' in df.columns:
    monthly = df.groupby(df['fecha_creacion'].dt.to_period('M'))[TARGET_COL].mean()
    monthly.plot(ax=axes[1], color='#e74c3c', marker='o')
    axes[1].set_title('Tasa de Incumplimiento por Mes')
    axes[1].set_ylabel('Tasa')
    axes[1].tick_params(axis='x', rotation=45)

# Por carrier (top 10)
if 'carrier' in df.columns:
    carrier_rate = df.groupby('carrier')[TARGET_COL].agg(['mean', 'count'])
    carrier_rate = carrier_rate[carrier_rate['count'] >= 100].sort_values('mean', ascending=True).tail(10)
    carrier_rate['mean'].plot(kind='barh', ax=axes[2], color='#e74c3c', alpha=0.7)
    axes[2].set_title('Tasa de Incumplimiento por Carrier (top 10)')
    axes[2].set_xlabel('Tasa')

plt.tight_layout()
plt.show()
print(f'\nDistribución target:')
print(df[TARGET_COL].value_counts(normalize=True).round(3))

## 3. Correlación Features vs Target

In [ ]:
# Correlación de features numéricas con el target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != TARGET_COL]

correlations = df[numeric_cols + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
correlations = correlations.sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 6))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in correlations]
correlations.plot(kind='bar', color=colors, alpha=0.7)
plt.title('Correlación de Features Numéricas con Target (Incumplimiento)')
plt.ylabel('Correlación de Pearson')
plt.axhline(0, color='black', linewidth=0.5)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('Top correlaciones positivas (incrementan riesgo):')
print(correlations[correlations > 0].head(5))
print('\nTop correlaciones negativas (reducen riesgo):')
print(correlations[correlations < 0].head(5))

## 4. Exploración PULSE Features

In [ ]:
pulse_numeric = ['num_ops', 'num_processes', 'num_legs', 'num_transfers',
                 'total_span_hours', 'picking_window_hours', 'complexity_score', 'span_ratio']
pulse_available = [c for c in pulse_numeric if c in df.columns]

if pulse_available:
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for i, col in enumerate(pulse_available[:8]):
        fail = df[df[TARGET_COL]==1][col].dropna()
        success = df[df[TARGET_COL]==0][col].dropna()
        
        axes[i].hist(success, bins=30, alpha=0.5, color='#2ecc71', label='Cumplió', density=True)
        axes[i].hist(fail, bins=30, alpha=0.5, color='#e74c3c', label='Incumplió', density=True)
        axes[i].set_title(col)
        axes[i].legend(fontsize=7)
    
    plt.suptitle('Distribución de PULSE Features: Cumplió vs Incumplió', fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print('PULSE features no disponibles en este dataset')

## 5. Análisis Post-Entrenamiento

In [ ]:
# Cargar modelo entrenado
try:
    model = joblib.load(MODELS_DIR / 'model_latest.pkl')
    import json
    with open(MODELS_DIR / 'features_latest.json') as f:
        feat_data = json.load(f)
    feature_names = feat_data['features']
    print(f'✓ Modelo cargado: {feat_data["version"]}')
    print(f'  Features: {len(feature_names)}')
except FileNotFoundError:
    print('⚠ Modelo no encontrado. Ejecutar 03_train_model.py primero.')
    model = None
    feature_names = []

In [ ]:
# Métricas de validación
if model:
    try:
        import json
        with open(MODELS_DIR / 'metrics_latest.json') as f:
            metrics = json.load(f)
        val = metrics.get('validation', {})
        print('Métricas en Validación:')
        for k, v in val.items():
            if isinstance(v, float):
                print(f'  {k:30s}: {v:.4f}')
    except Exception as e:
        print(f'Error cargando métricas: {e}')

In [ ]:
# SHAP - Importancia global
if model and (OUTPUTS_DIR / 'shap_global.png').exists():
    from IPython.display import Image, display
    display(Image(OUTPUTS_DIR / 'shap_global.png'))
    
    # Tabla de importancia
    importance_df = pd.read_csv(OUTPUTS_DIR / 'feature_importance.csv')
    print('Top 15 features por importancia SHAP:')
    display(importance_df.head(15).style.bar(subset=['mean_abs_shap'], color='#e74c3c'))
else:
    print('Ejecutar 04_evaluate.py para generar gráficos SHAP')

## 6. Debug de Predicciones Específicas

In [ ]:
# ── ZONA DE EXPERIMENTACIÓN LIBRE ──────────────────────────────────────────
# Analizar una predicción específica de alto riesgo

# Ejemplo: encontrar órdenes con mayor risk score
# (requiere que hayas corrido inference sobre el dataset)

# X_test = np.load(OUTPUTS_DIR / 'X_test.npy', allow_pickle=True)
# y_test = np.load(OUTPUTS_DIR / 'y_test.npy', allow_pickle=True)
# 
# if model:
#     y_prob = model.predict_proba(X_test)[:, 1]
#     # Top 10 de mayor riesgo
#     top_risk_idx = np.argsort(y_prob)[::-1][:10]
#     print('Top 10 predicciones de mayor riesgo:')
#     for i in top_risk_idx:
#         print(f'  idx={i} | risk_score={y_prob[i]:.3f} | actual={y_test[i]}')

print('Descomenta el código de arriba para analizar predicciones específicas')

## 7. Preparación para Event Log
*(Ejecutar cuando lleguen los datos)*

In [ ]:
# Cuando llegue el event log, explorar aquí antes de incorporarlo

# event_log_path = Path('../data/event_log.csv')
# 
# if event_log_path.exists():
#     events = pd.read_csv(event_log_path, parse_dates=['event_timestamp'])
#     print(f'Event log cargado: {len(events):,} eventos')
#     print(f'\nDistribución de tipos de eventos:')
#     print(events['event_type'].value_counts())
#     
#     print(f'\nRango temporal: {events["event_timestamp"].min()} → {events["event_timestamp"].max()}')
#     print(f'Orders únicas: {events["order_id"].nunique():,}')
#     
#     # Eventos por orden
#     events_per_order = events.groupby('order_id').size()
#     print(f'\nEventos por orden - stats:')
#     print(events_per_order.describe())
# else:
#     print(f'Event log no encontrado en {event_log_path}')
#     print('Cuando llegue: copiar a data/event_log.csv y ejecutar esta celda')

print('Event log pendiente. Código listo para ejecutar cuando llegue.')

In [ ]:
# Validar calidad del event log cuando llegue
# 
# Checklist:
# □ ¿Cuántos orders del training set tienen eventos?
# □ ¿Los timestamps están en orden cronológico?
# □ ¿Hay gaps de silencio mayores a 72h?
# □ ¿Los tipos de eventos coinciden con el glosario operativo?
# 
# if event_log_path.exists():
#     # Match con dataset de entrenamiento
#     train_orders = set(df['order_id'].unique())
#     event_orders = set(events['order_id'].unique())
#     match_pct = len(train_orders & event_orders) / len(train_orders) * 100
#     print(f'Match eventos ↔ órdenes de training: {match_pct:.1f}%')
#     print(f'Mínimo recomendado para features temporales: 70%')

print('Validación de event log pendiente.')